# **Core Analytics — Dosier de encargos de cliente**

### ***Consultoría en Ciencia de Datos y Machine Learning · Del dato a la decisión***

---

Cada banco de datos se presenta una **petición de asesoramiento** de una empresa o administración a Core Analytics. Para cada encargo se describe el cliente y la cuestión planteada, los datos aportados con su diccionario de variables (identificador, significado y unidades), las **preguntas que el cliente desea ver respondidas** y las **líneas abiertas**, formuladas como políticas, protocolos de funcionamiento y herramientas de despliegue sobre las que la firma tiene libertad para innovar.

Solicitudes de asesoramiento

---

> **Cómo usar este cuaderno.** Cada encargo ocupa una celda de texto con el enunciado íntegro y una celda de código lista para cargar su fichero. Ejecuta primero la celda de **Configuración** y después ve directamente al encargo que te haya sido asignado.

## Índice de encargos

### Parte A — Encargos exploratorios (no supervisados)

- [01 · Consorcio de Atención Primaria Vega Norte](#01-consorcio-de-atencion-primaria-vega-norte) · `01_salud_pacientes.csv`
- [02 · Cooperativa Agraria San Isidro](#02-cooperativa-agraria-san-isidro) · `02_agricultura_parcelas.csv`
- [03 · Observatorio de Inclusión Financiera Meridiano](#03-observatorio-de-inclusion-financiera-meridiano) · `03_economia_hogares.csv`
- [04 · Agencia Regional de Medio Ambiente (ARMA)](#04-agencia-regional-de-medio-ambiente-arma) · `04_medioambiente_estaciones.csv`
- [05 · Instituto de Biodiversidad y Museo de Ciencias Naturales](#05-instituto-de-biodiversidad-y-museo-de-ciencias-naturales) · `05_biologia_morfometria.csv`

### Parte B — Encargos predictivos (supervisados)

- [SU-01 · Dirección de Atención Primaria Vega Norte](#su-01-direccion-de-atencion-primaria-vega-norte) · `SU01_salud_diabetes.csv` · objetivo: `diabetes`
- [SU-02 · ARMA, Servicio de Calidad del Aire](#su-02-arma-servicio-de-calidad-del-aire) · `SU02_medioambiente_ozono.csv` · objetivo: `alerta_ozono`
- [SU-03 · Autoridad Portuaria](#su-03-autoridad-portuaria) · `SU03_biologia_invasora.csv` · objetivo: `especie_invasora`
- [SU-04 · Financiera Castnor](#su-04-financiera-castnor) · `SU04_economia_impago.csv` · objetivo: `impago`
- [SU-05 · Cooperativa Agraria San Isidro y Agroseguro](#su-05-cooperativa-agraria-san-isidro-y-agroseguro) · `SU05_agricultura_plaga.csv` · objetivo: `perdida_cosecha`
- [SU-06 · Hospital Universitario Costa](#su-06-hospital-universitario-costa) · `SU06_salud_reingreso.csv` · objetivo: `reingreso_30d`

---

## Nota metodológica común a todos los encargos

1. **El cliente no pide un modelo: pide una decisión.** Las *preguntas* se responden con evidencia; las *líneas abiertas* se responden con criterio. Un cuaderno que solo optimiza una métrica no ha respondido al encargo.
2. **Escala dentro del `Pipeline`, nunca antes de partir.** Ajustar un `StandardScaler` sobre el conjunto completo filtra información del test.
3. **Ningún encargo trae partición predefinida.** Construye la tuya, estratificada, y no la toques hasta la evaluación final.
4. **Cuatro de los once encargos contienen datos de personas** (salud, hogares, crédito, reingresos). En ellos, la exactitud global no es la métrica relevante: evalúa por subgrupos y razona el coste asimétrico del error.


---

## ⚙️ Configuración del entorno

Ejecuta esta celda una sola vez al abrir el cuaderno. Ajusta `DATA_DIR` a la carpeta donde tengas los CSV: en Colab lo habitual es montar Google Drive; en local, basta con una ruta relativa.

In [1]:
#@title Configuración del entorno
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", palette="deep")

# --- Origen de los datos -------------------------------------------------
# Los ficheros del dosier están publicados en GitHub. No hay que montar
# Drive ni subir nada: se descargan al vuelo la primera vez y se guardan
# en una caché local para que las siguientes ejecuciones sean instantáneas.
BASE_URL = (
    "https://raw.githubusercontent.com/jmsocuellamos/MachineLearningBook/"
    "refs/heads/main/ml-book/bloque8/"
)
CACHE_DIR = Path("datos_cache")
CACHE_DIR.mkdir(exist_ok=True)
# -------------------------------------------------------------------------

RANDOM_STATE = 42

FICHEROS = {
    "01": "01_salud_pacientes.csv",
    "02": "02_agricultura_parcelas.csv",
    "03": "03_economia_hogares.csv",
    "04": "04_medioambiente_estaciones.csv",
    "05": "05_biologia_morfometria.csv",
    "SU-01": "SU01_salud_diabetes.csv",
    "SU-02": "SU02_medioambiente_ozono.csv",
    "SU-03": "SU03_biologia_invasora.csv",
    "SU-04": "SU04_economia_impago.csv",
    "SU-05": "SU05_agricultura_plaga.csv",
    "SU-06": "SU06_salud_reingreso.csv",
}


def cargar(nombre: str, cache: bool = True) -> pd.DataFrame:
    """Carga un fichero del dosier desde GitHub y muestra un resumen mínimo.

    Acepta el nombre del CSV ("SU01_salud_diabetes.csv") o el código del
    encargo ("SU-01"). Con cache=True guarda una copia local en CACHE_DIR.
    """
    fichero = FICHEROS.get(nombre.upper(), nombre)
    if not fichero.endswith(".csv"):
        raise ValueError(
            f"No reconozco '{nombre}'. Usa un nombre de fichero o uno de: "
            f"{', '.join(FICHEROS)}"
        )

    local = CACHE_DIR / fichero
    if cache and local.exists():
        df, origen = pd.read_csv(local), "caché local"
    else:
        try:
            df = pd.read_csv(BASE_URL + fichero)
        except Exception as e:
            raise ConnectionError(
                f"No he podido descargar {fichero}.\n"
                f"URL: {BASE_URL + fichero}\n"
                f"Comprueba tu conexión o descarga el fichero a {CACHE_DIR}/\n"
                f"Detalle: {e}"
            ) from e
        origen = "GitHub"
        if cache:
            df.to_csv(local, index=False)

    print(f"{fichero}  ←  {origen}")
    print(f"  {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"  Valores faltantes: {df.isna().sum().sum():,}")
    print(f"  Duplicados: {df.duplicated().sum():,}")

    ids = [c for c in df.columns if c.lower().startswith("id_") or c.lower() == "id"]
    if ids:
        print(f"  ⚠️  Identificador(es) no predictivo(s): {', '.join(ids)} → excluir del modelo")
    return df


print(f"Entorno listo. {len(FICHEROS)} encargos disponibles desde GitHub.")
print(f"Caché local: {CACHE_DIR.resolve()}")

Entorno listo. 11 encargos disponibles desde GitHub.
Caché local: /content/datos_cache


---

# Parte A — Encargos exploratorios

Los cinco encargos siguientes **no tienen variable objetivo**. El cliente no pide predecir: pide *descubrir estructura*. Son problemas de **aprendizaje no supervisado** (segmentación, reducción de dimensión, detección de atípicos), y su evaluación no puede apoyarse en una métrica de acierto: hay que argumentar la validez del resultado.

<a name="01-consorcio-de-atencion-primaria-vega-norte"></a>
## 01 · Consorcio de Atención Primaria Vega Norte

El Consorcio de Atención Primaria Vega Norte es una entidad sanitaria pública integrada en el servicio regional de salud que gestiona la atención primaria de una comarca de unos 180.000 habitantes a través de 12 centros de salud y varios consultorios rurales, con cerca de 90 médicos de familia y 70 profesionales de enfermería. Su población envejece y la carga de enfermedades crónicas (diabetes, hipertensión, patología cardiovascular) crece año tras año, presionando las agendas y el gasto. El Consorcio ha aprobado un plan estratégico orientado a la **salud preventiva y comunitaria** y acaba de completar la digitalización de las historias clínicas, pero no dispone de un equipo de análisis de datos propio: hasta ahora explota la información de forma agregada y poco accionable. Acude a Core Analytics para empezar a sacar partido a esos datos sin tener que crear una unidad interna de la noche a la mañana.

> **📁 Fichero de datos**
> `01_salud_pacientes.csv`

### La cuestión

El Consorcio quiere pasar de un modelo reactivo a uno **preventivo y personalizado**. Dispone de una muestra de pacientes con datos antropométricos, analíticos y de hábitos de vida, pero carece de una forma sistemática de agruparlos. Solicita a Core Analytics que **descubra perfiles de pacientes clínicamente coherentes** que permitan diseñar programas de prevención dirigidos y asignar recursos de enfermería de forma más eficiente. No existe una etiqueta previa: el objetivo es exploratorio.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `edad` | Edad | años |
| `peso_kg` | Peso corporal | kg |
| `altura_cm` | Altura | cm |
| `imc` | Índice de masa corporal | kg/m² |
| `perimetro_cintura_cm` | Perímetro de cintura | cm |
| `perimetro_cadera_cm` | Perímetro de cadera | cm |
| `presion_sistolica` | Presión arterial sistólica | mmHg |
| `presion_diastolica` | Presión arterial diastólica | mmHg |
| `frecuencia_cardiaca_reposo` | Frecuencia cardiaca en reposo | lpm |
| `glucosa_ayunas` | Glucemia en ayunas | mg/dL |
| `hba1c` | Hemoglobina glicosilada | % |
| `colesterol_total` | Colesterol total | mg/dL |
| `hdl` | Colesterol HDL | mg/dL |
| `ldl` | Colesterol LDL | mg/dL |
| `trigliceridos` | Triglicéridos | mg/dL |
| `creatinina` | Creatinina sérica | mg/dL |
| `filtrado_glomerular` | Filtrado glomerular estimado | mL/min/1.73m² |
| `pcr` | Proteína C reactiva | mg/L |
| `vitamina_d` | Vitamina D (25-OH) | ng/mL |
| `hemoglobina` | Hemoglobina | g/dL |
| `ejercicio_sem_h` | Ejercicio físico semanal | horas/semana |
| `pasos_dia` | Pasos diarios | pasos |
| `horas_sueno` | Horas de sueño | horas/día |
| `consumo_alcohol_sem` | Consumo de alcohol | UBE/semana |
| `cigarrillos_dia` | Consumo de tabaco | cigarrillos/día |
| `frutas_verduras_dia` | Raciones de fruta y verdura | raciones/día |
| `sal_diaria_g` | Sal diaria estimada | g/día |
| `nivel_estres` | Nivel de estrés autopercibido | escala 0-10 |
| `sexo` | Sexo biológico | categórico (M/F) |
| `antecedentes_familiares` | Antecedentes familiares de enf. metabólica | categórico (si/no) |
| `actividad_laboral` | Nivel de actividad laboral | categórico (sedentario/ligero/activo) |

### Preguntas que el cliente desea responder

1. ¿Cuántos perfiles de paciente diferenciados existen y qué proporción de la población representa cada uno?
2. Para cada perfil, ¿cuáles son los valores típicos de IMC, presión sistólica, glucosa, HDL y triglicéridos?
3. ¿Qué perfil presenta el peor patrón metabólico combinado (glucosa alta, triglicéridos altos y HDL bajo)?
4. ¿Qué perfil concentra el mayor consumo de tabaco y el menor nivel de ejercicio?
5. ¿Existe un perfil "joven de riesgo", definido por hábitos más que por la edad?
6. ¿Cómo se distribuyen el sexo y la edad dentro de cada perfil?
7. ¿Qué proporción de pacientes queda en perfiles esencialmente saludables que solo requieren seguimiento rutinario?
8. ¿Qué perfil debería ser la prioridad número uno para una intervención inmediata y con qué justificación clínica?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir un **protocolo de citación y seguimiento** diferenciado por perfil (frecuencia de revisiones y profesional asignado).
- Establecer una **política de asignación de programas preventivos** (actividad física, deshabituación tabáquica, control metabólico) a cada perfil.
- Proponer una **herramienta de despliegue**: un cuadro de mando para enfermería que asigne automáticamente cada paciente a su perfil al introducir sus datos.
- Fijar un **procedimiento de revisión periódica** de los perfiles y de reasignación de pacientes a lo largo del tiempo.

In [2]:
# 01 · Consorcio de Atención Primaria Vega Norte
df_01 = cargar("01_salud_pacientes.csv")
df_01.head()

01_salud_pacientes.csv  ←  GitHub
  13,000 filas × 32 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_paciente → excluir del modelo


,id_paciente,edad,peso_kg,altura_cm,imc,perimetro_cintura_cm,perimetro_cadera_cm,presion_sistolica,presion_diastolica,frecuencia_cardiaca_reposo,glucosa_ayunas,hba1c,colesterol_total,hdl,ldl,trigliceridos,creatinina,filtrado_glomerular,pcr,vitamina_d,hemoglobina,ejercicio_sem_h,pasos_dia,horas_sueno,consumo_alcohol_sem,cigarrillos_dia,frutas_verduras_dia,sal_diaria_g,nivel_estres,sexo,antecedentes_familiares,actividad_laboral
0,PAC000000,29,72.2,166,26.2,83,97,136,72,90,100,4.6,182,65,145,106,1.02,99,0.1,17.1,10.2,4,10300,7.5,18,20,1,2.9,9,M,no,sedentario
1,PAC000001,75,79.9,191,21.9,106,108,142,74,79,148,6.9,199,33,155,192,1.13,98,11.0,41.1,12.9,0,3344,7.6,13,7,0,14.0,10,F,si,ligero
2,PAC000002,90,73.0,154,30.8,98,104,179,103,85,129,5.4,259,33,143,292,0.54,76,4.7,33.6,17.6,4,2105,3.6,1,14,1,12.3,10,F,no,ligero
3,PAC000003,69,107.1,150,47.6,83,104,118,80,68,104,6.1,255,49,95,105,0.40,81,0.7,35.0,13.8,3,6194,7.1,19,6,1,17.2,5,F,no,sedentario
4,PAC000004,28,55.3,169,19.4,84,93,103,79,48,72,4.7,139,69,68,113,1.06,118,1.0,17.9,15.5,11,12614,8.9,3,7,5,7.2,2,F,no,ligero


<a name="02-cooperativa-agraria-san-isidro"></a>
## 02 · Cooperativa Agraria San Isidro

La Cooperativa Agraria San Isidro, fundada en 1968, agrupa a unos 850 socios que cultivan en conjunto alrededor de 6.500 hectáreas de cereal, viñedo, olivar y hortícola. Además de comercializar la producción, abastece a sus socios de insumos (semilla, fertilizantes, fitosanitarios) y les ofrece asesoramiento a través de un pequeño servicio agronómico de apenas tres técnicos para todo el territorio. La cooperativa afronta una doble presión: la creciente variabilidad climática y el encarecimiento de los insumos por un lado, y las exigencias regulatorias y de sostenibilidad (reducción del uso de fitosanitarios y de agua) por otro. Hoy el asesoramiento es prácticamente uniforme para todos los socios, lo que resulta ineficiente. La cooperativa quiere profesionalizar su servicio técnico apoyándose en los datos de suelo y clima que ya recopila, y por eso recurre a Core Analytics.

> **📁 Fichero de datos**
> `02_agricultura_parcelas.csv`

### La cuestión

Pide a Core Analytics que **clasifique las parcelas en tipologías agronómicas homogéneas** para emitir recomendaciones diferenciadas de cultivo, abonado y riego, y negociar compras de insumos por grupo.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `ph_suelo` | pH del suelo | escala pH |
| `materia_organica_pct` | Materia orgánica | % |
| `carbono_organico_pct` | Carbono orgánico | % |
| `nitrogeno_ppm` | Nitrógeno disponible | ppm (mg/kg) |
| `fosforo_ppm` | Fósforo disponible | ppm (mg/kg) |
| `potasio_ppm` | Potasio disponible | ppm (mg/kg) |
| `calcio_ppm` | Calcio intercambiable | ppm |
| `magnesio_ppm` | Magnesio intercambiable | ppm |
| `azufre_ppm` | Azufre disponible | ppm |
| `conductividad_electrica` | Conductividad eléctrica | dS/m |
| `cic` | Capacidad de intercambio catiónico | cmol/kg |
| `caliza_activa_pct` | Caliza activa | % |
| `arcilla_pct` | Textura: arcilla | % |
| `arena_pct` | Textura: arena | % |
| `limo_pct` | Textura: limo | % |
| `humedad_suelo_pct` | Humedad del suelo | % |
| `capacidad_campo_pct` | Capacidad de campo | % |
| `profundidad_suelo_cm` | Profundidad útil del suelo | cm |
| `pendiente_pct` | Pendiente del terreno | % |
| `altitud_m` | Altitud | m |
| `precipitacion_anual_mm` | Precipitación anual | mm |
| `temp_media_c` | Temperatura media anual | °C |
| `temp_min_invierno_c` | Temperatura mínima de invierno | °C |
| `temp_max_verano_c` | Temperatura máxima de verano | °C |
| `radiacion_anual` | Radiación solar anual | kWh/m² |
| `evapotranspiracion_mm` | Evapotranspiración potencial | mm |
| `indice_aridez` | Índice de aridez (P/ETP) | adimensional |
| `orientacion` | Orientación dominante | categórico (N/S/E/O) |
| `tipo_suelo` | Tipo de suelo predominante | categórico (franco/arcilloso/arenoso/calcareo) |

### Preguntas que el cliente desea responder

1. ¿Cuántas tipologías de parcela existen y qué proporción representa cada una?
2. ¿Qué tipología presenta mayores carencias de nitrógeno, fósforo o potasio?
3. ¿Qué tipología combina baja materia orgánica y baja humedad (suelos más pobres y secos)?
4. ¿Qué rangos de pH caracterizan a cada grupo y cuáles requerirían corrección (encalado o acidificación)?
5. ¿Qué tipología se sitúa a mayor altitud y menor temperatura, condicionando los cultivos viables?
6. ¿Qué grupo recibe menos precipitación y depende más del riego?
7. ¿Hay parcelas que, por sus valores, no encajan en ninguna tipología y requieren visita técnica individual?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir un **plan de abonado y de enmiendas** diferenciado por tipología.
- Establecer una **política de compra agrupada de insumos** por grupo de parcelas.
- Crear un **protocolo de asesoramiento al socio**: ficha por tipología con recomendaciones de cultivo, riego y abonado.
- **Herramienta de despliegue**: un formulario o app donde el técnico introduzca los datos de una parcela nueva y obtenga su tipología y recomendaciones.

In [3]:
# 02 · Cooperativa Agraria San Isidro
df_02 = cargar("02_agricultura_parcelas.csv")
df_02.head()

02_agricultura_parcelas.csv  ←  GitHub
  13,000 filas × 30 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_parcela → excluir del modelo


,id_parcela,ph_suelo,materia_organica_pct,carbono_organico_pct,nitrogeno_ppm,fosforo_ppm,potasio_ppm,calcio_ppm,magnesio_ppm,azufre_ppm,conductividad_electrica,cic,caliza_activa_pct,arcilla_pct,arena_pct,limo_pct,humedad_suelo_pct,capacidad_campo_pct,profundidad_suelo_cm,pendiente_pct,altitud_m,precipitacion_anual_mm,temp_media_c,temp_min_invierno_c,temp_max_verano_c,radiacion_anual,evapotranspiracion_mm,indice_aridez,orientacion,tipo_suelo
0,PAR000000,5.3,4.9,1.56,18,24,212,720,209,11.2,0.67,23.0,1.8,38.4,44.1,5.0,18.6,18.0,84,8.6,591,588,19.4,2.9,30.0,1580,925,1.06,N,franco
1,PAR000001,6.1,4.6,1.55,41,28,305,3325,179,12.4,0.75,24.7,0.8,26.2,39.2,36.6,34.3,33.2,88,0.0,849,818,15.5,6.1,31.3,1716,829,0.89,S,franco
2,PAR000002,6.0,1.6,0.81,6,15,169,459,61,20.1,0.63,24.0,18.2,9.1,44.9,47.3,20.7,40.1,15,24.1,440,819,16.7,5.2,35.5,1763,1235,0.38,N,calcareo
3,PAR000003,7.8,0.3,1.86,12,3,40,893,138,22.2,1.54,15.6,29.3,55.8,37.3,22.9,8.3,45.9,72,19.0,420,241,18.4,4.8,33.6,1837,1079,0.10,O,arenoso
4,PAR000004,5.4,5.4,0.46,28,34,243,3042,304,25.4,0.60,32.1,5.3,20.3,52.8,52.0,25.6,54.3,77,7.6,878,677,9.1,5.8,36.9,2020,613,0.82,E,arcilloso


<a name="03-observatorio-de-inclusion-financiera-meridiano"></a>
## 03 · Observatorio de Inclusión Financiera Meridiano

El Observatorio de Inclusión Financiera Meridiano es la obra social de una antigua caja de ahorros, hoy convertida en fundación bancaria sin ánimo de lucro. Su misión es estudiar y mejorar la **salud financiera de los hogares**, especialmente de los colectivos vulnerables. Publica informes periódicos, financia programas de educación financiera y colabora con servicios sociales municipales y entidades del tercer sector. No comercializa productos por sí mismo, pero su trabajo influye en el diseño de productos responsables de la entidad matriz y en recomendaciones de política pública. Cuenta con economistas y trabajadores sociales, pero su capacidad de análisis cuantitativo es limitada y suele encargarlo fuera. Acude a Core Analytics para fundamentar con evidencia su próximo informe y sus programas.

> **📁 Fichero de datos**
> `03_economia_hogares.csv`

### La cuestión

Encarga una **segmentación de hogares** que combine ingresos, gasto, ahorro y endeudamiento, con especial atención a la detección de hogares vulnerables, para diseñar productos y medidas de educación financiera adecuados a cada situación.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `ingreso_mensual_eur` | Ingreso neto mensual del hogar | €/mes |
| `ingreso_principal_eur` | Ingreso del sustentador principal | €/mes |
| `ingreso_secundario_eur` | Otros ingresos del hogar | €/mes |
| `ratio_gasto_ingreso` | Gasto sobre ingreso | adimensional |
| `gasto_mensual_eur` | Gasto total mensual | €/mes |
| `gasto_vivienda_pct` | Peso del gasto en vivienda | % del gasto |
| `gasto_alimentacion_pct` | Peso del gasto en alimentación | % del gasto |
| `gasto_suministros_pct` | Peso del gasto en suministros | % del gasto |
| `gasto_transporte_pct` | Peso del gasto en transporte | % del gasto |
| `gasto_ocio_pct` | Peso del gasto en ocio | % del gasto |
| `gasto_educacion_pct` | Peso del gasto en educación | % del gasto |
| `gasto_salud_pct` | Peso del gasto en salud | % del gasto |
| `ahorro_pct` | Tasa de ahorro sobre ingreso | % |
| `ahorro_acumulado_eur` | Ahorro acumulado | € |
| `deuda_eur` | Deuda total pendiente | € |
| `deuda_consumo_eur` | Deuda de consumo | € |
| `cuota_deuda_mensual_eur` | Cuota mensual de deuda | €/mes |
| `ratio_endeudamiento_pct` | Carga de deuda sobre ingreso | % |
| `num_miembros` | Miembros del hogar | personas |
| `num_perceptores` | Perceptores de ingresos | personas |
| `num_menores` | Menores en el hogar | personas |
| `edad_sustentador` | Edad del sustentador principal | años |
| `num_tarjetas` | Tarjetas de crédito | nº |
| `num_productos_bancarios` | Productos bancarios contratados | nº |
| `antiguedad_cliente_anos` | Antigüedad bancaria | años |
| `patrimonio_neto_eur` | Patrimonio neto estimado | € |
| `regimen_vivienda` | Régimen de tenencia de la vivienda | categórico (propiedad/alquiler/hipoteca) |
| `nivel_estudios` | Nivel de estudios del sustentador | categórico (basico/medio/superior) |
| `situacion_laboral` | Situación laboral del sustentador | categórico (ocupado/parado/jubilado/inactivo) |

### Preguntas que el cliente desea responder

1. ¿Cuántos segmentos de hogar existen y qué proporción de la población representa cada uno?
2. ¿Qué segmento presenta mayor endeudamiento relativo (deuda frente a ingresos) y menor ahorro?
3. ¿Qué segmento dedica un mayor porcentaje del gasto a vivienda, quedándose sin margen?
4. ¿Qué caracteriza a los hogares con mayor capacidad de ahorro (ingresos, tamaño, régimen de vivienda)?
5. ¿Cómo se relaciona el régimen de tenencia (alquiler frente a propiedad) con la vulnerabilidad?
6. ¿Qué segmento combina ingresos bajos, ahorro nulo y deuda elevada (vulnerabilidad severa)?
7. ¿Qué tamaño tiene el colectivo vulnerable y qué peso relativo sobre el total?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir una **política de productos financieros responsables** por segmento (microahorro, refinanciación, cuentas básicas).
- Establecer un **protocolo de detección y derivación** de hogares vulnerables a servicios sociales o de educación financiera.
- Diseñar un **programa de educación financiera** con itinerarios adaptados a cada segmento.
- **Herramienta de despliegue**: un panel para que los asesores clasifiquen un hogar y reciban la recomendación y el itinerario correspondientes.

In [4]:
# 03 · Observatorio de Inclusión Financiera Meridiano
df_03 = cargar("03_economia_hogares.csv")
df_03.head()

03_economia_hogares.csv  ←  GitHub
  13,000 filas × 30 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_hogar → excluir del modelo


,id_hogar,ingreso_mensual_eur,ingreso_principal_eur,ingreso_secundario_eur,ratio_gasto_ingreso,gasto_mensual_eur,gasto_vivienda_pct,gasto_alimentacion_pct,gasto_suministros_pct,gasto_transporte_pct,gasto_ocio_pct,gasto_educacion_pct,gasto_salud_pct,ahorro_pct,ahorro_acumulado_eur,deuda_eur,deuda_consumo_eur,cuota_deuda_mensual_eur,ratio_endeudamiento_pct,num_miembros,num_perceptores,num_menores,edad_sustentador,num_tarjetas,num_productos_bancarios,antiguedad_cliente_anos,patrimonio_neto_eur,regimen_vivienda,nivel_estudios,situacion_laboral
0,HOG000000,4742,1924,2231,1.05,4979,34.3,16.7,23.4,12.9,11.3,8.8,9.9,21.4,17498,19888,15532,237,17.3,1,3,2,20,1,1,27.1,249628,propiedad,superior,jubilado
1,HOG000001,4335,3150,2244,0.84,3641,29.4,33.0,13.3,3.5,4.1,3.3,10.5,27.1,13255,6243,10892,300,22.8,4,2,2,22,3,1,13.1,131746,propiedad,medio,ocupado
2,HOG000002,7752,5489,0,0.32,2481,8.0,17.7,6.4,13.9,22.0,9.5,12.5,47.4,107408,0,3059,415,0.0,1,2,0,80,4,8,24.7,407070,propiedad,medio,jubilado
3,HOG000003,700,1044,1578,1.28,896,65.4,31.3,11.2,14.6,2.8,6.2,4.8,0.0,6472,16802,12528,654,59.8,2,0,0,43,0,1,12.9,-26121,alquiler,medio,inactivo
4,HOG000004,1029,500,856,0.87,895,46.7,30.9,11.6,7.5,7.2,2.6,10.5,1.7,0,13145,6297,1092,48.8,5,1,0,47,1,1,5.6,161348,alquiler,basico,parado


<a name="04-agencia-regional-de-medio-ambiente-arma"></a>
## 04 · Agencia Regional de Medio Ambiente (ARMA)

La Agencia Regional de Medio Ambiente (ARMA) es un organismo público autónomo responsable de vigilar la calidad ambiental de la región y de velar por el cumplimiento de las directivas europeas de aire limpio. Opera una red de unas 45 estaciones de medición que registran contaminantes de forma continua, además de coordinar las respuestas ante episodios de contaminación y de informar a la ciudadanía. Su plantilla está formada sobre todo por ingenieros y técnicos ambientales; su capacidad de ciencia de datos es escasa y la explotación de la información se limita a comprobar superaciones de umbrales legales. La red ha crecido por acumulación histórica, sin un rediseño global, y ARMA sospecha que hay redundancias y zonas mal cubiertas. Recurre a Core Analytics para racionalizarla con criterio técnico.

> **📁 Fichero de datos**
> `04_medioambiente_estaciones.csv`

### La cuestión

El encargo es agrupar las estaciones en **tipologías diferenciadas** según los contaminantes que registran y su contexto (tráfico, población), para evitar redundancias, detectar zonas mal cubiertas y comunicar mejor a la ciudadanía.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `no2_ugm3` | NO₂ (media anual) | µg/m³ |
| `no2_max_ugm3` | NO₂ (máximo horario) | µg/m³ |
| `nox_ugm3` | NOx (media anual) | µg/m³ |
| `pm10_ugm3` | PM10 (media anual) | µg/m³ |
| `pm10_max_ugm3` | PM10 (máximo diario) | µg/m³ |
| `pm25_ugm3` | PM2.5 (media anual) | µg/m³ |
| `pm25_max_ugm3` | PM2.5 (máximo diario) | µg/m³ |
| `o3_ugm3` | O₃ (media anual) | µg/m³ |
| `o3_max_ugm3` | O₃ (máximo octohorario) | µg/m³ |
| `so2_ugm3` | SO₂ (media anual) | µg/m³ |
| `co_mgm3` | CO (media anual) | mg/m³ |
| `benceno_ugm3` | Benceno | µg/m³ |
| `nh3_ugm3` | Amoniaco (NH₃) | µg/m³ |
| `ruido_db` | Nivel de ruido medio | dB |
| `trafico_indice` | Índice de intensidad de tráfico | 0-100 |
| `intensidad_trafico` | Intensidad media de tráfico | veh/día |
| `pct_vehiculos_pesados` | Vehículos pesados | % |
| `densidad_poblacion` | Densidad de población del entorno | hab/km² |
| `dist_industria_km` | Distancia a polígono industrial | km |
| `dist_via_principal_m` | Distancia a vía principal | m |
| `superficie_verde_pct` | Zonas verdes en el entorno | % |
| `temp_media_c` | Temperatura media | °C |
| `humedad_pct` | Humedad relativa media | % |
| `viento_kmh` | Velocidad media del viento | km/h |
| `precipitacion_mm` | Precipitación anual | mm |
| `radiacion_solar` | Radiación solar media | W/m² |
| `altitud_m` | Altitud de la estación | m |
| `num_dias_superacion` | Días/año con superación de umbral | días |
| `entorno` | Entorno geográfico | categórico (costa/interior) |

### Preguntas que el cliente desea responder

1. ¿Cuántos tipos de estación existen y cuántas estaciones hay en cada uno?
2. ¿Qué tipo está dominado por NO₂ y CO (tráfico) frente a cuál por SO₂ (industrial)?
3. ¿Qué tipo registra los mayores niveles de ozono y en qué contexto (fondo rural, baja densidad)?
4. ¿Qué tipo concentra las mayores PM10 y PM2.5?
5. ¿Coincide la tipología por contaminantes con el contexto de tráfico y densidad de población?
6. ¿Hay estaciones cuyo perfil de contaminantes no concuerde con su entorno (posible mala ubicación)?
7. ¿Qué tipos están sobrerrepresentados y cuáles infrarrepresentados en la red actual?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Proponer una **política de rediseño de la red**: estaciones a mantener, reubicar o añadir.
- Definir un **protocolo de actuación por tipo de estación** ante superación de umbrales (avisos, restricciones de tráfico).
- Establecer un **protocolo de comunicación pública** diferenciado por tipo de zona.
- **Herramienta de despliegue**: un cuadro de mando que asigne cada estación a su tipo y muestre el estado de la red.

In [5]:
# 04 · Agencia Regional de Medio Ambiente (ARMA)
df_04 = cargar("04_medioambiente_estaciones.csv")
df_04.head()

04_medioambiente_estaciones.csv  ←  GitHub
  13,000 filas × 30 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_estacion → excluir del modelo


,id_estacion,no2_ugm3,no2_max_ugm3,nox_ugm3,pm10_ugm3,pm10_max_ugm3,pm25_ugm3,pm25_max_ugm3,o3_ugm3,o3_max_ugm3,so2_ugm3,co_mgm3,benceno_ugm3,nh3_ugm3,ruido_db,trafico_indice,intensidad_trafico,pct_vehiculos_pesados,densidad_poblacion,dist_industria_km,dist_via_principal_m,superficie_verde_pct,temp_media_c,humedad_pct,viento_kmh,precipitacion_mm,radiacion_solar,altitud_m,num_dias_superacion,entorno
0,EST000000,33,149,83,27,61,23,42,75,51,10.3,0.20,1.78,0.5,68.0,67,34839,8.4,8628,8.1,539,30.4,17.8,62,20.1,591,443,258,18,interior
1,EST000001,2,36,22,17,19,2,22,82,194,3.9,0.10,0.53,19.1,39.6,0,100,3.2,20,15.0,1734,57.4,16.2,64,12.6,504,335,337,25,interior
2,EST000002,79,205,97,37,130,33,5,56,56,0.0,1.29,2.74,4.7,77.9,45,30839,10.7,6425,14.6,5,0.1,20.0,47,21.7,838,567,967,46,interior
3,EST000003,68,199,140,42,70,23,31,49,53,6.9,1.61,1.26,0.5,73.2,100,60863,11.5,12000,12.3,5,0.0,19.2,45,19.2,1273,664,613,65,costa
4,EST000004,49,185,113,50,102,27,23,25,80,31.7,1.49,2.56,11.0,75.2,82,19011,32.4,7341,1.3,5,14.0,15.1,56,12.9,608,494,0,66,costa


<a name="05-instituto-de-biodiversidad-y-museo-de-ciencias-naturales"></a>
## 05 · Instituto de Biodiversidad y Museo de Ciencias Naturales

El Instituto de Biodiversidad y Museo de Ciencias Naturales es un organismo de investigación de titularidad pública, vinculado a varias universidades, que custodia colecciones científicas con cientos de miles de ejemplares, entre ellas una importante colección entomológica. En él trabajan taxónomos, ecólogos y personal de conservación, y desde hace unos años desarrolla un ambicioso proyecto de digitalización de sus fondos que ha generado grandes tablas de medidas morfométricas. Sin embargo, el equipo investigador domina la sistemática clásica pero no las técnicas cuantitativas de análisis multivariante, y buena parte de esos datos digitalizados está infrautilizada. El Instituto quiere extraer valor científico de ese esfuerzo de digitalización y, en concreto, explorar la hipótesis de que algunos taxones encierran especies crípticas. Por eso colabora con Core Analytics.

> **📁 Fichero de datos**
> `05_biologia_morfometria.csv`

### La cuestión

Solicita un análisis exploratorio de agrupamiento que revele cuántos **grupos morfológicos diferenciados** existen y qué medidas los separan, como paso previo a un estudio taxonómico más profundo.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `longitud_total_mm` | Longitud total del cuerpo | mm |
| `longitud_elitro_mm` | Longitud del élitro | mm |
| `anchura_max_mm` | Anchura máxima | mm |
| `anchura_pronoto_mm` | Anchura del pronoto | mm |
| `anchura_cabeza_mm` | Anchura de la cabeza | mm |
| `altura_cuerpo_mm` | Altura del cuerpo | mm |
| `longitud_antena_mm` | Longitud de la antena | mm |
| `num_segmentos_antena` | Segmentos antenales | nº |
| `longitud_pata_anterior_mm` | Longitud pata anterior | mm |
| `longitud_pata_posterior_mm` | Longitud pata posterior | mm |
| `longitud_femur_mm` | Longitud del fémur | mm |
| `longitud_tibia_mm` | Longitud de la tibia | mm |
| `longitud_tarso_mm` | Longitud del tarso | mm |
| `distancia_interocular_mm` | Distancia interocular | mm |
| `diametro_ojo_mm` | Diámetro del ojo | mm |
| `longitud_mandibula_mm` | Longitud de la mandíbula | mm |
| `num_estrias_elitro` | Estrías del élitro | nº |
| `num_puntos_pronoto` | Densidad de puntuación del pronoto | puntos |
| `longitud_espolon_mm` | Longitud del espolón tibial | mm |
| `peso_mg` | Peso | mg |
| `ratio_largo_ancho` | Relación longitud/anchura | adimensional |
| `ratio_antena_cuerpo` | Relación antena/cuerpo | adimensional |
| `ratio_pata_cuerpo` | Relación pata posterior/cuerpo | adimensional |
| `area_dorsal_mm2` | Área dorsal estimada | mm² |
| `volumen_estimado_mm3` | Volumen corporal estimado | mm³ |
| `longitud_ala_mm` | Longitud del ala | mm |
| `envergadura_mm` | Envergadura alar | mm |
| `coloracion` | Coloración dominante | categórico (negro/marron/metalico/rojizo) |

### Preguntas que el cliente desea responder

1. ¿Cuántos grupos morfológicos distintos sugieren los datos?
2. ¿Qué proporción de especímenes cae en cada grupo?
3. ¿Qué medidas (longitud, peso, ratio, segmentos antenales) discriminan mejor entre grupos?
4. ¿Qué grupos son más similares entre sí y cuáles más distintos?
5. ¿Existen especímenes intermedios o ambiguos y qué porcentaje representan?
6. ¿Hay algún grupo con un tamaño o una forma marcadamente distinta (posible especie no descrita)?
7. ¿Los grupos difieren más en tamaño global o en proporciones corporales (forma)?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Proponer un **protocolo de muestreo y medición estandarizado** para futuras campañas (qué medidas tomar y cómo).
- Definir un **procedimiento de etiquetado y conservación** de los especímenes representativos de cada grupo en la colección.
- Establecer una **política de priorización** para estudios genéticos confirmatorios (qué grupos analizar primero).
- **Herramienta de despliegue**: una ficha digital por grupo y un sistema que asigne nuevos especímenes a su grupo al registrarlos.

In [6]:
# 05 · Instituto de Biodiversidad y Museo de Ciencias Naturales
df_05 = cargar("05_biologia_morfometria.csv")
df_05.head()

05_biologia_morfometria.csv  ←  GitHub
  13,000 filas × 29 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_especimen → excluir del modelo


,id_especimen,longitud_total_mm,longitud_elitro_mm,anchura_max_mm,anchura_pronoto_mm,anchura_cabeza_mm,altura_cuerpo_mm,longitud_antena_mm,num_segmentos_antena,longitud_pata_anterior_mm,longitud_pata_posterior_mm,longitud_femur_mm,longitud_tibia_mm,longitud_tarso_mm,distancia_interocular_mm,diametro_ojo_mm,longitud_mandibula_mm,num_estrias_elitro,num_puntos_pronoto,longitud_espolon_mm,peso_mg,ratio_largo_ancho,ratio_antena_cuerpo,ratio_pata_cuerpo,area_dorsal_mm2,volumen_estimado_mm3,longitud_ala_mm,envergadura_mm,coloracion
0,ESP000000,9.40,5.77,2.96,2.69,1.73,1.19,6.58,8,4.80,5.24,3.36,6.06,1.75,1.44,1.16,1.71,8,48,1.31,17,3.18,0.700,0.557,21.7,16.6,8.81,20.58,marron
1,ESP000001,22.18,14.14,10.92,8.74,5.92,7.19,11.83,12,3.79,16.80,5.54,3.45,3.35,2.10,0.72,3.56,13,61,0.10,300,2.03,0.533,0.757,188.9,870.7,18.10,47.12,metalico
2,ESP000002,11.67,6.78,4.26,3.54,2.17,5.08,7.52,8,5.09,2.00,4.31,1.13,2.82,1.97,0.89,0.61,6,92,0.26,86,2.74,0.644,0.171,38.8,126.3,9.55,23.36,negro
3,ESP000003,19.23,11.64,9.52,7.50,5.21,5.98,11.63,10,4.30,13.84,2.39,4.79,2.97,2.09,0.34,2.53,12,75,0.85,270,2.02,0.605,0.720,142.8,547.4,17.00,43.52,negro
4,ESP000004,19.40,12.27,9.23,7.11,4.91,7.83,10.55,10,6.08,13.05,2.93,3.49,5.45,2.10,0.61,1.83,12,51,0.98,229,2.10,0.544,0.673,139.7,701.0,18.13,45.49,metalico


---

# Parte B — Encargos predictivos

Los seis encargos siguientes **sí tienen variable objetivo**, señalada en negrita en cada diccionario. Todos son problemas de **clasificación binaria**, pero con costes de error muy distintos entre sí: conviene decidir el umbral de decisión *antes* de mirar los resultados, y justificarlo con el criterio del cliente, no con el AUC.

<a name="su-01-direccion-de-atencion-primaria-vega-norte"></a>
## SU-01 · Dirección de Atención Primaria Vega Norte

Es la Dirección de Atención Primaria del mismo Consorcio Vega Norte descrito en el encargo 01: la unidad directiva responsable de la planificación asistencial, la cartera de servicios y los indicadores de calidad de los 12 centros de salud. Gestiona un presupuesto ajustado y agendas saturadas, con la diabetes tipo 2 como uno de sus principales retos de cronicidad por su prevalencia y su coste a largo plazo (complicaciones renales, cardiovasculares, oftalmológicas). La Dirección sabe que muchos casos se diagnostican tarde y querría adelantarse usando la información analítica que ya genera de forma rutinaria, sin incurrir en pruebas caras adicionales. Tras la buena experiencia exploratoria del encargo de perfiles, da un paso más y pide a Core Analytics una herramienta predictiva concreta.

> **📁 Fichero de datos**
> `SU01_salud_diabetes.csv`  ·  **Variable objetivo:** `diabetes`

### La cuestión

Quiere implantar un **cribado proactivo de diabetes** para citar antes a los pacientes de mayor riesgo. Encarga un modelo que estime el riesgo a partir de parámetros numéricos habituales, sea interpretable y se pueda ajustar al coste de no detectar un caso frente al de una falsa alarma.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `edad` | Edad | años |
| `peso_kg` | Peso corporal | kg |
| `altura_cm` | Altura | cm |
| `imc` | Índice de masa corporal | kg/m² |
| `perimetro_cintura_cm` | Perímetro de cintura | cm |
| `perimetro_cadera_cm` | Perímetro de cadera | cm |
| `indice_cintura_cadera` | Índice cintura/cadera | adimensional |
| `presion_sistolica` | Presión arterial sistólica | mmHg |
| `presion_diastolica` | Presión arterial diastólica | mmHg |
| `frecuencia_cardiaca` | Frecuencia cardiaca | lpm |
| `glucosa_ayunas` | Glucemia en ayunas | mg/dL |
| `glucosa_postprandial` | Glucemia posprandial | mg/dL |
| `hba1c` | Hemoglobina glicosilada | % |
| `insulina_uUml` | Insulina en ayunas | µU/mL |
| `homa_ir` | Índice HOMA-IR | adimensional |
| `peptido_c` | Péptido C | ng/mL |
| `colesterol_total` | Colesterol total | mg/dL |
| `hdl` | Colesterol HDL | mg/dL |
| `ldl` | Colesterol LDL | mg/dL |
| `trigliceridos` | Triglicéridos | mg/dL |
| `acido_urico` | Ácido úrico | mg/dL |
| `creatinina` | Creatinina sérica | mg/dL |
| `filtrado_glomerular` | Filtrado glomerular estimado | mL/min/1.73m² |
| `alt_got` | ALT (GPT) | U/L |
| `ggt` | GGT | U/L |
| `pcr` | Proteína C reactiva | mg/L |
| `vitamina_d` | Vitamina D (25-OH) | ng/mL |
| `hemoglobina` | Hemoglobina | g/dL |
| `num_embarazos` | Número de embarazos | nº |
| `antecedentes_familiares` | Antecedentes familiares de diabetes | binaria (0/1) |
| `actividad_fisica_h_sem` | Actividad física semanal | horas/semana |
| `horas_sueno` | Horas de sueño | horas/día |
| **`diabetes`** | **Diabetes (objetivo)** | binario (0/1) |

### Preguntas que el cliente desea responder

1. ¿Qué fiabilidad alcanza el cribado en términos de sensibilidad, especificidad y AUC?
2. ¿Qué variables son las más determinantes y en qué orden de importancia?
3. ¿A partir de qué niveles de glucosa en ayunas y de HbA1c se dispara el riesgo?
4. ¿Qué umbral de decisión captura al menos el 90 % de los casos y cuántas falsas alarmas implica?
5. ¿Qué proporción de la población quedaría clasificada como "alto riesgo" con ese umbral?
6. ¿El rendimiento del modelo es homogéneo por sexo y por tramos de edad?
7. ¿Cuántas pruebas confirmatorias adicionales generaría el cribado a lo largo de un año?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir un **protocolo de cribado por niveles de riesgo** (alto: cita preferente; medio: control en 6 meses; bajo: rutina).
- Establecer una **política de periodicidad** de recálculo del riesgo y de repetición de analíticas.
- **Herramienta de despliegue**: integrar el cálculo de riesgo en la historia clínica electrónica, mostrando el nivel al abrir la ficha del paciente.
- Fijar un **plan de monitorización y gobernanza** del modelo en producción (seguimiento de aciertos y reentrenamiento).

In [7]:
# SU-01 · Dirección de Atención Primaria Vega Norte
df_su01 = cargar("SU01_salud_diabetes.csv")
print("\nReparto del objetivo:")
print(df_su01["diabetes"].value_counts(normalize=True).round(3))
df_su01.head()

SU01_salud_diabetes.csv  ←  GitHub
  13,000 filas × 34 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_paciente → excluir del modelo

Reparto del objetivo:
diabetes
0    0.825
1    0.175
Name: proportion, dtype: float64


,id_paciente,edad,peso_kg,altura_cm,imc,perimetro_cintura_cm,perimetro_cadera_cm,indice_cintura_cadera,presion_sistolica,presion_diastolica,frecuencia_cardiaca,glucosa_ayunas,glucosa_postprandial,hba1c,insulina_uUml,homa_ir,peptido_c,colesterol_total,hdl,ldl,trigliceridos,acido_urico,creatinina,filtrado_glomerular,alt_got,ggt,pcr,vitamina_d,hemoglobina,num_embarazos,antecedentes_familiares,actividad_fisica_h_sem,horas_sueno,diabetes
0,DBS000000,62,98.7,170,34.2,120,122,0.98,108,80,76,119,165,8.3,14.1,4.14,4.09,236,43,127,195,4.2,0.99,111,34,69,4.7,27.5,14.2,0,1,1,5.3,0
1,DBS000001,38,71.9,194,19.1,85,110,0.77,133,67,70,109,120,4.4,13.0,3.50,2.57,189,50,162,70,5.0,0.76,91,24,32,0.1,27.9,15.0,1,0,4,5.6,0
2,DBS000002,45,65.0,181,19.8,80,102,0.78,122,62,72,60,70,4.0,2.2,0.33,2.14,171,53,112,73,2.2,0.71,130,34,5,1.3,26.1,15.7,2,0,5,8.6,0
3,DBS000003,83,73.2,165,26.9,89,93,0.96,121,87,60,74,85,4.2,5.2,0.95,2.60,256,60,128,48,2.9,0.59,105,30,64,2.3,37.2,11.8,2,0,5,7.9,0
4,DBS000004,52,107.9,151,47.3,105,111,0.95,155,94,72,116,172,7.2,20.2,5.79,3.28,247,39,184,198,9.4,0.86,98,43,10,2.4,14.0,16.6,0,1,3,6.7,1


<a name="su-02-arma-servicio-de-calidad-del-aire"></a>
## SU-02 · ARMA, Servicio de Calidad del Aire

Es el Servicio de Calidad del Aire dentro de ARMA (descrita en 04), la unidad encargada del día a día de la red de vigilancia y, sobre todo, de la gestión de los episodios de contaminación. Este servicio es quien, en verano, debe decidir cuándo activar los avisos a la población por ozono troposférico, un contaminante que no se emite directamente sino que se forma con el sol y el calor a partir de precursores. Hoy actúa de forma reactiva: avisa cuando los niveles ya están altos, lo que deja poco margen a la población vulnerable y a las medidas de tráfico. El servicio dispone de un buen histórico meteorológico y de contaminantes, pero carece de un modelo que le permita anticiparse. Encarga a Core Analytics dar ese salto hacia la predicción.

> **📁 Fichero de datos**
> `SU02_medioambiente_ozono.csv`  ·  **Variable objetivo:** `alerta_ozono`

### La cuestión

Quiere **anticipar con un día de margen los episodios de alerta por ozono** para activar avisos y medidas de tráfico, priorizando no dejar pasar episodios reales.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `temp_max_c` | Temperatura máxima | °C |
| `temp_min_c` | Temperatura mínima | °C |
| `temp_media_c` | Temperatura media | °C |
| `amplitud_termica_c` | Amplitud térmica | °C |
| `radiacion_solar_wm2` | Radiación solar | W/m² |
| `radiacion_uv_indice` | Índice UV | 0-11 |
| `horas_sol` | Horas de sol | horas |
| `humedad_pct` | Humedad relativa | % |
| `humedad_min_pct` | Humedad relativa mínima | % |
| `viento_kmh` | Velocidad del viento | km/h |
| `viento_max_kmh` | Racha máxima de viento | km/h |
| `direccion_viento_deg` | Dirección del viento | grados |
| `presion_atm_hpa` | Presión atmosférica | hPa |
| `no2_ugm3` | Dióxido de nitrógeno | µg/m³ |
| `no_ugm3` | Monóxido de nitrógeno | µg/m³ |
| `nox_ugm3` | Óxidos de nitrógeno (NOx) | µg/m³ |
| `cov_precursores` | COV precursores (índice) | µg/m³ |
| `co_mgm3` | Monóxido de carbono | mg/m³ |
| `so2_ugm3` | Dióxido de azufre | µg/m³ |
| `pm10_ugm3` | PM10 | µg/m³ |
| `o3_dia_previo_ugm3` | Ozono del día previo | µg/m³ |
| `o3_media_movil_3d` | Ozono medio móvil 3 días | µg/m³ |
| `temp_dia_previo_c` | Temperatura máxima del día previo | °C |
| `gradiente_temp_c` | Gradiente térmico vertical | °C |
| `altura_capa_mezcla_m` | Altura de la capa de mezcla | m |
| `indice_estabilidad` | Índice de estabilidad atmosférica | -10..10 |
| `precipitacion_mm` | Precipitación del día | mm |
| `horas_desde_lluvia` | Horas desde la última lluvia | horas |
| **`alerta_ozono`** | **Día de alerta por ozono (objetivo)** | binario (0/1) |

### Preguntas que el cliente desea responder

1. ¿Con qué fiabilidad puede anticiparse un día de alerta con un día de antelación?
2. ¿Qué combinación de temperatura máxima, radiación y horas de sol marca el umbral de riesgo?
3. ¿Qué papel juegan el viento y la humedad como factores atenuantes?
4. ¿Qué umbral de decisión evita perder más de un 5–10 % de los episodios reales?
5. ¿Cuántas falsas alertas al año supondría ese umbral?
6. ¿Aportan el NO₂ y la presión atmosférica capacidad adicional para afinar la predicción?
7. ¿El modelo funciona igual en distintos rangos de temperatura o estaciones del año?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir un **protocolo de avisos escalonado** (verde/amarillo/rojo) con acciones asociadas a cada nivel.
- Establecer una **política de medidas de tráfico y de protección a población vulnerable** por nivel de alerta.
- **Herramienta de despliegue**: un servicio que cada tarde calcule la previsión del día siguiente y dispare avisos automáticos (web, SMS, app).
- Fijar un **plan de recalibración estacional** del sistema y su procedimiento de mantenimiento.

In [8]:
# SU-02 · ARMA, Servicio de Calidad del Aire
df_su02 = cargar("SU02_medioambiente_ozono.csv")
print("\nReparto del objetivo:")
print(df_su02["alerta_ozono"].value_counts(normalize=True).round(3))
df_su02.head()

SU02_medioambiente_ozono.csv  ←  GitHub
  13,000 filas × 30 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_registro → excluir del modelo

Reparto del objetivo:
alerta_ozono
0    0.867
1    0.133
Name: proportion, dtype: float64


,id_registro,temp_max_c,temp_min_c,temp_media_c,amplitud_termica_c,radiacion_solar_wm2,radiacion_uv_indice,horas_sol,humedad_pct,humedad_min_pct,viento_kmh,viento_max_kmh,direccion_viento_deg,presion_atm_hpa,no2_ugm3,no_ugm3,nox_ugm3,cov_precursores,co_mgm3,so2_ugm3,pm10_ugm3,o3_dia_previo_ugm3,o3_media_movil_3d,temp_dia_previo_c,gradiente_temp_c,altura_capa_mezcla_m,indice_estabilidad,precipitacion_mm,horas_desde_lluvia,alerta_ozono
0,OZN000000,31.1,22.6,26.8,8.5,689,8,14.0,21,8,9.1,3,205,1013,17,1,61,69,0.10,3.7,44,102,131,34.5,0.67,1062,0.5,0.0,152,1
1,OZN000001,18.8,15.1,17.0,3.7,313,8,10.0,81,34,7.9,39,24,1011,28,13,127,49,0.66,2.7,34,45,55,27.1,1.08,1650,2.2,0.0,51,0
2,OZN000002,24.3,6.6,15.4,17.7,703,8,3.4,88,55,7.1,30,316,1021,54,5,24,57,0.92,11.6,19,36,99,28.5,0.78,1535,-6.9,2.8,141,0
3,OZN000003,26.2,6.6,16.4,19.6,500,4,8.3,84,49,6.8,23,197,1016,42,21,56,51,0.87,5.2,36,145,129,19.6,0.82,1627,3.3,0.6,0,0
4,OZN000004,15.2,19.0,17.1,-3.8,242,4,3.7,81,76,15.1,27,303,1013,34,25,61,40,0.65,10.7,24,145,111,23.1,0.51,1064,-1.0,0.0,0,1


<a name="su-03-autoridad-portuaria"></a>
## SU-03 · Autoridad Portuaria

La Autoridad Portuaria gestiona uno de los principales puertos comerciales del país, por el que pasan cada año millones de toneladas de mercancía y cientos de miles de contenedores. Su Servicio de Inspección de Sanidad Exterior es responsable, junto con aduanas y el ministerio competente, del control fitosanitario de las importaciones, una primera línea de defensa frente a la entrada de plagas y especies invasoras que pueden causar daños ecológicos y económicos enormes. El problema es de escala: el volumen de tráfico es altísimo y el número de inspectores, limitado, por lo que solo puede revisarse en profundidad una fracción de los envíos. Entre los organismos interceptados aparece con frecuencia un insecto difícil de distinguir a simple vista de una especie autóctona inofensiva. La Autoridad busca en Core Analytics una herramienta que ayude a sus inspectores a decidir, deprisa, qué ejemplares merecen análisis prioritario.

> **📁 Fichero de datos**
> `SU03_biologia_invasora.csv`  ·  **Variable objetivo:** `especie_invasora`

### La cuestión

Quiere una **herramienta de apoyo a la inspección** que, con medidas morfométricas rápidas, señale los especímenes sospechosos para análisis prioritario, con muy pocos falsos negativos.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `longitud_total_mm` | Longitud total del cuerpo | mm |
| `longitud_elitro_mm` | Longitud del élitro | mm |
| `anchura_max_mm` | Anchura máxima | mm |
| `anchura_pronoto_mm` | Anchura del pronoto | mm |
| `anchura_cabeza_mm` | Anchura de la cabeza | mm |
| `altura_cuerpo_mm` | Altura del cuerpo | mm |
| `peso_mg` | Peso | mg |
| `longitud_antena_mm` | Longitud de la antena | mm |
| `num_segmentos_antena` | Segmentos antenales | nº |
| `longitud_pata_posterior_mm` | Longitud pata posterior | mm |
| `longitud_femur_mm` | Longitud del fémur | mm |
| `longitud_tibia_mm` | Longitud de la tibia | mm |
| `longitud_ala_mm` | Longitud del ala | mm |
| `envergadura_mm` | Envergadura alar | mm |
| `num_manchas` | Manchas en los élitros | nº |
| `diametro_mancha_mm` | Diámetro medio de mancha | mm |
| `num_estrias_elitro` | Estrías del élitro | nº |
| `distancia_interocular_mm` | Distancia interocular | mm |
| `diametro_ojo_mm` | Diámetro del ojo | mm |
| `longitud_mandibula_mm` | Longitud de la mandíbula | mm |
| `ratio_largo_ancho` | Relación longitud/anchura | adimensional |
| `ratio_antena_cuerpo` | Relación antena/cuerpo | adimensional |
| `ratio_ala_cuerpo` | Relación ala/cuerpo | adimensional |
| `area_dorsal_mm2` | Área dorsal estimada | mm² |
| `volumen_estimado_mm3` | Volumen corporal estimado | mm³ |
| `densidad_setas` | Densidad de vellosidad (setas) | nº/mm² |
| `intensidad_color` | Intensidad de color (0=claro,255=oscuro) | 0-255 |
| **`especie_invasora`** | **Espécimen de especie invasora (objetivo)** | binario (0/1) |

### Preguntas que el cliente desea responder

1. ¿Se puede distinguir la especie invasora con suficiente fiabilidad a partir de la morfometría?
2. ¿Qué medidas son las más discriminantes y qué valores son característicos de la invasora?
3. ¿Qué umbral garantiza detectar al menos el 95 % de los invasores y cuántos autóctonos de más se revisarían?
4. ¿Qué porcentaje del flujo de inspección quedaría marcado como "sospechoso" con ese umbral?
5. ¿Hay zonas de solape donde el modelo es poco fiable y conviene confirmación obligatoria?
6. ¿Aportan el número de manchas o el ratio largo/ancho información más allá del tamaño corporal?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir un **protocolo de inspección de dos fases**: cribado morfométrico rápido y confirmación de laboratorio para los sospechosos.
- Establecer una **política de priorización y trazabilidad** de los especímenes marcados.
- **Herramienta de despliegue**: una app de inspección donde el agente introduzca las medidas y obtenga al instante el nivel de sospecha.
- Fijar un **procedimiento de actualización** del sistema cuando se confirmen nuevos casos (realimentación).

In [9]:
# SU-03 · Autoridad Portuaria
df_su03 = cargar("SU03_biologia_invasora.csv")
print("\nReparto del objetivo:")
print(df_su03["especie_invasora"].value_counts(normalize=True).round(3))
df_su03.head()

SU03_biologia_invasora.csv  ←  GitHub
  13,000 filas × 29 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_especimen → excluir del modelo

Reparto del objetivo:
especie_invasora
0    0.842
1    0.158
Name: proportion, dtype: float64


,id_especimen,longitud_total_mm,longitud_elitro_mm,anchura_max_mm,anchura_pronoto_mm,anchura_cabeza_mm,altura_cuerpo_mm,peso_mg,longitud_antena_mm,num_segmentos_antena,longitud_pata_posterior_mm,longitud_femur_mm,longitud_tibia_mm,longitud_ala_mm,envergadura_mm,num_manchas,diametro_mancha_mm,num_estrias_elitro,distancia_interocular_mm,diametro_ojo_mm,longitud_mandibula_mm,ratio_largo_ancho,ratio_antena_cuerpo,ratio_ala_cuerpo,area_dorsal_mm2,volumen_estimado_mm3,densidad_setas,intensidad_color,especie_invasora
0,INV000000,17.49,10.80,6.70,5.61,2.05,4.18,58,7.13,12,4.14,4.35,4.27,13.87,34.44,1,0.10,9,1.38,0.90,1.22,2.61,0.408,0.793,91.4,244.9,5,45,0
1,INV000001,16.58,10.28,6.04,3.73,4.74,3.43,186,11.97,8,8.52,3.14,4.45,14.97,35.98,13,1.44,12,1.91,0.97,2.56,2.75,0.722,0.903,78.1,171.7,23,213,1
2,INV000002,19.14,11.97,7.50,5.43,3.85,4.59,124,5.57,9,14.72,4.98,4.17,11.25,30.00,4,0.85,10,1.79,1.26,1.79,2.55,0.291,0.588,112.0,329.4,7,73,0
3,INV000003,25.09,15.60,6.56,6.64,5.42,6.30,305,11.23,11,9.42,5.54,5.18,14.51,35.58,13,1.23,9,2.63,0.74,3.06,3.82,0.448,0.578,128.4,518.5,25,178,1
4,INV000004,14.72,9.15,5.23,5.39,3.98,1.00,98,8.75,10,6.00,1.93,2.45,9.02,23.27,2,0.89,8,1.64,0.71,0.66,2.81,0.594,0.613,60.0,38.5,12,204,0


<a name="su-04-financiera-castnor"></a>
## SU-04 · Financiera Castnor

Financiera Castnor es una entidad de crédito al consumo con cerca de 25 años de actividad, que concede miles de préstamos de importe pequeño y medio cada mes, tanto a través de su web como de una red de comercios asociados (electrodomésticos, automoción, viajes). Opera en un sector muy competitivo y regulado, en el que el margen depende de afinar el riesgo: aprobar de más dispara la morosidad, pero rechazar de más cede negocio a la competencia. Su departamento de riesgos utiliza desde hace años un sistema de scoring basado en reglas fijas, que se ha quedado corto frente a métodos más modernos. La dirección quiere modernizar la decisión con aprendizaje automático, pero con una condición innegociable: que las decisiones sean **explicables**, tanto de cara al cliente como ante el supervisor. Con ese encargo llega a Core Analytics.

> **📁 Fichero de datos**
> `SU04_economia_impago.csv`  ·  **Variable objetivo:** `impago`

### La cuestión

Quiere **reforzar su política de riesgos** anticipando qué solicitudes acabarán en impago, equilibrando el coste de un crédito fallido con el de rechazar a un buen cliente, mediante un modelo explicable y un análisis de los factores de riesgo.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `edad` | Edad del solicitante | años |
| `ingresos_mensuales_eur` | Ingresos netos mensuales | €/mes |
| `ingresos_conyuge_eur` | Ingresos del cónyuge | €/mes |
| `importe_prestamo_eur` | Importe solicitado | € |
| `plazo_meses` | Plazo de amortización | meses |
| `cuota_mensual_eur` | Cuota mensual estimada | €/mes |
| `ratio_endeudamiento` | Carga de la cuota sobre ingresos | % |
| `ratio_cuota_ingreso` | Cuota sobre ingreso total del hogar | % |
| `antiguedad_empleo_anos` | Antigüedad en el empleo | años |
| `antiguedad_cliente_anos` | Antigüedad como cliente | años |
| `num_prestamos_previos` | Préstamos anteriores | nº |
| `num_prestamos_activos` | Préstamos activos | nº |
| `num_impagos_previos` | Impagos anteriores | nº |
| `dias_mora_max_historico` | Días de mora máximos (histórico) | días |
| `score_credito` | Puntuación crediticia interna | 300-850 |
| `score_externo` | Puntuación crediticia externa | 0-1000 |
| `saldo_medio_cuenta_eur` | Saldo medio en cuenta | € |
| `num_productos_contratados` | Productos contratados | nº |
| `patrimonio_declarado_eur` | Patrimonio declarado | € |
| `num_consultas_buro_6m` | Consultas al buró (6 meses) | nº |
| `tasa_interes_pct` | Tipo de interés aplicado | % |
| `gastos_fijos_mensuales_eur` | Gastos fijos mensuales | €/mes |
| `finalidad` | Finalidad del préstamo | categórico (vivienda/coche/consumo/negocio/estudios) |
| `tipo_empleo` | Tipo de empleo | categórico (fijo/temporal/autonomo/funcionario) |
| `regimen_vivienda` | Régimen de vivienda | categórico (propiedad/alquiler/hipoteca) |
| `nivel_estudios` | Nivel de estudios | categórico (basico/medio/superior) |
| `sector_actividad` | Sector de actividad | categórico (servicios/industria/construccion/agrario/publico) |
| `tiene_avalista` | Dispone de avalista | categórico (si/no) |
| **`impago`** | **Impago del préstamo (objetivo)** | binario (0/1) |

### Preguntas que el cliente desea responder

1. ¿Qué fiabilidad alcanza la predicción de impago?
2. ¿Qué variables explican mejor el riesgo (score, ratio de endeudamiento, tipo de empleo…) y en qué medida?
3. ¿Cómo varía el riesgo según la finalidad del préstamo y el tipo de empleo?
4. ¿A partir de qué ratio de endeudamiento, o por debajo de qué score, se dispara el impago?
5. Dado un coste por crédito fallido y otro por cliente bueno rechazado, ¿qué umbral de aprobación los equilibra?
6. ¿Qué tasa de aprobación resultaría de aplicar ese umbral?
7. ¿El modelo trata de forma equivalente a solicitantes de distinta edad o régimen de vivienda?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir una **política de decisión por tramos de riesgo** (aprobar / revisar manualmente / denegar), con condiciones por tramo.
- Establecer un **protocolo de explicación de la decisión** al cliente y de gestión de reclamaciones.
- **Herramienta de despliegue**: un motor de scoring integrado en el sistema de originación que devuelva riesgo y motivo en tiempo real.
- Fijar un **plan de gobernanza del modelo**: seguimiento del impago real frente al previsto y reentrenamiento periódico.

In [10]:
# SU-04 · Financiera Castnor
df_su04 = cargar("SU04_economia_impago.csv")
print("\nReparto del objetivo:")
print(df_su04["impago"].value_counts(normalize=True).round(3))
df_su04.head()

SU04_economia_impago.csv  ←  GitHub
  13,000 filas × 30 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_prestamo → excluir del modelo

Reparto del objetivo:
impago
0    0.801
1    0.199
Name: proportion, dtype: float64


,id_prestamo,edad,ingresos_mensuales_eur,ingresos_conyuge_eur,importe_prestamo_eur,plazo_meses,cuota_mensual_eur,ratio_endeudamiento,ratio_cuota_ingreso,antiguedad_empleo_anos,antiguedad_cliente_anos,num_prestamos_previos,num_prestamos_activos,num_impagos_previos,dias_mora_max_historico,score_credito,score_externo,saldo_medio_cuenta_eur,num_productos_contratados,patrimonio_declarado_eur,num_consultas_buro_6m,tasa_interes_pct,gastos_fijos_mensuales_eur,finalidad,tipo_empleo,regimen_vivienda,nivel_estudios,sector_actividad,tiene_avalista,impago
0,PRE000000,53,1711,1216,11858,39,328,19.2,11.2,4.2,8.2,3,1,0,19,664,464,9798,0,52951,4,2.0,891,coche,fijo,propiedad,medio,construccion,no,0
1,PRE000001,44,2491,622,6372,12,573,23.0,18.4,11.2,11.6,1,1,1,33,730,520,-341,6,72535,2,5.8,1975,negocio,temporal,alquiler,superior,industria,si,0
2,PRE000002,69,2788,754,11320,51,240,8.6,6.8,10.8,0.0,1,3,0,17,670,763,914,3,0,2,2.7,1512,consumo,fijo,propiedad,medio,construccion,no,0
3,PRE000003,55,1829,522,11277,23,530,29.0,22.5,0.0,0.0,2,2,1,20,553,621,9258,0,73682,0,2.0,251,consumo,temporal,hipoteca,medio,industria,no,1
4,PRE000004,43,3042,1572,11090,65,184,6.0,4.0,0.5,1.5,2,1,1,45,696,715,-2000,3,34097,1,9.5,829,consumo,fijo,alquiler,basico,publico,si,0


<a name="su-05-cooperativa-agraria-san-isidro-y-agroseguro"></a>
## SU-05 · Cooperativa Agraria San Isidro y Agroseguro

Este encargo lo plantean conjuntamente la Cooperativa Agraria San Isidro (presentada en NS-02) y su entidad aseguradora, Agroseguro, una mutua especializada en seguros agrarios. La cooperativa quiere proteger la renta de sus socios reduciendo las pérdidas por plagas, y la aseguradora quiere tarificar con más justicia y anticipar su siniestralidad. Ambas comparten un mismo interés: pasar de una gestión reactiva —tratar cuando el daño ya es visible— a una **preventiva y basada en el riesgo**. Disponen de datos agroclimáticos y de manejo de las parcelas, pero ni la cooperativa ni la mutua cuentan con perfiles de ciencia de datos. Por eso encargan conjuntamente el proyecto a Core Analytics, con la idea de que sus resultados sirvan tanto para decidir tratamientos como para ajustar primas.

> **📁 Fichero de datos**
> `SU05_agricultura_plaga.csv`  ·  **Variable objetivo:** `perdida_cosecha`

### La cuestión

Quieren **anticipar qué parcelas sufrirán una pérdida relevante por plaga** para priorizar tratamientos preventivos y para que la aseguradora ajuste sus primas con criterio.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `temp_media_c` | Temperatura media del periodo | °C |
| `temp_max_c` | Temperatura máxima | °C |
| `temp_min_c` | Temperatura mínima | °C |
| `humedad_pct` | Humedad relativa | % |
| `humedad_suelo_pct` | Humedad del suelo | % |
| `precipitacion_mm` | Precipitación del periodo | mm |
| `precipitacion_acum_30d_mm` | Precipitación acumulada 30 días | mm |
| `horas_humectacion_foliar` | Horas de humectación foliar | horas |
| `ndvi` | Índice de vegetación NDVI | 0-1 |
| `ndwi` | Índice de agua NDWI | -1..1 |
| `indice_estres_hidrico` | Índice de estrés hídrico | 0-1 |
| `dias_desde_tratamiento` | Días desde el último tratamiento | días |
| `num_tratamientos_campana` | Tratamientos en la campaña | nº |
| `densidad_plantas_m2` | Densidad de plantación | plantas/m² |
| `ph_suelo` | pH del suelo | escala pH |
| `materia_organica_pct` | Materia orgánica del suelo | % |
| `nitrogeno_ppm` | Nitrógeno disponible | ppm |
| `conductividad_suelo` | Conductividad eléctrica del suelo | dS/m |
| `altitud_m` | Altitud | m |
| `pendiente_pct` | Pendiente del terreno | % |
| `dist_foco_plaga_km` | Distancia al foco de plaga más próximo | km |
| `trampas_capturas_num` | Capturas en trampas de monitoreo | nº |
| `grados_dia_acumulados` | Grados-día acumulados | °C·día |
| `indice_presion_plaga` | Índice de presión de plaga | 0-100 |
| `cobertura_vegetal_pct` | Cobertura vegetal | % |
| `edad_cultivo_dias` | Edad del cultivo | días |
| `tipo_cultivo` | Cultivo | categórico (trigo/maiz/vid/olivo/horticola/citricos) |
| `sistema_riego` | Sistema de riego | categórico (secano/goteo/aspersion) |
| `uso_fitosanitario` | Uso de fitosanitarios | categórico (si/no) |
| `variedad_resistente` | Variedad resistente | categórico (si/no) |
| `manejo` | Tipo de manejo | categórico (convencional/ecologico/integrado) |
| **`perdida_cosecha`** | **Pérdida de cosecha por plaga (objetivo)** | binario (0/1) |

### Preguntas que el cliente desea responder

1. ¿Se puede predecir qué parcelas sufrirán pérdida por plaga y con qué fiabilidad?
2. ¿Qué condiciones (humedad, temperatura, días desde tratamiento, NDVI) elevan más el riesgo?
3. ¿Difieren el riesgo y sus causas entre tipos de cultivo (vid, olivo, hortícola…)?
4. ¿Qué papel juegan el sistema de riego y el uso de fitosanitarios?
5. ¿A partir de cuántos días desde el último tratamiento crece de forma marcada el riesgo?
6. ¿Qué umbral de riesgo identifica las parcelas a tratar capturando la mayoría de las pérdidas?
7. ¿Cuántas parcelas habría que tratar preventivamente con ese umbral?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir un **protocolo de tratamiento preventivo dirigido** por nivel de riesgo (umbral y momento de actuación).
- Establecer una **política de tarificación del seguro agrario** según el riesgo estimado de cada parcela.
- **Herramienta de despliegue**: un mapa o cuadro de mando de riesgo por parcela, actualizado por campaña para los técnicos de campo.
- Fijar un **procedimiento de aviso temprano** a los socios cuando una parcela supere el umbral de riesgo.

In [11]:
# SU-05 · Cooperativa Agraria San Isidro y Agroseguro
df_su05 = cargar("SU05_agricultura_plaga.csv")
print("\nReparto del objetivo:")
print(df_su05["perdida_cosecha"].value_counts(normalize=True).round(3))
df_su05.head()

SU05_agricultura_plaga.csv  ←  GitHub
  13,000 filas × 33 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_parcela → excluir del modelo

Reparto del objetivo:
perdida_cosecha
0    0.832
1    0.168
Name: proportion, dtype: float64


,id_parcela,temp_media_c,temp_max_c,temp_min_c,humedad_pct,humedad_suelo_pct,precipitacion_mm,precipitacion_acum_30d_mm,horas_humectacion_foliar,ndvi,ndwi,indice_estres_hidrico,dias_desde_tratamiento,num_tratamientos_campana,densidad_plantas_m2,ph_suelo,materia_organica_pct,nitrogeno_ppm,conductividad_suelo,altitud_m,pendiente_pct,dist_foco_plaga_km,trampas_capturas_num,grados_dia_acumulados,indice_presion_plaga,cobertura_vegetal_pct,edad_cultivo_dias,tipo_cultivo,sistema_riego,uso_fitosanitario,variedad_resistente,manejo,perdida_cosecha
0,CSC000000,18.6,22.8,16.7,46,14.6,37,106,8.8,0.76,0.11,0.41,51,0,194,6.8,2.8,8,0.77,425,21.6,11.8,0,986,51,73,146,olivo,aspersion,no,no,ecologico,0
1,CSC000001,27.5,28.5,9.0,46,10.6,72,51,12.1,0.52,0.20,0.48,0,4,163,5.6,1.6,21,0.11,266,18.1,0.2,42,1106,22,77,100,olivo,goteo,si,no,convencional,0
2,CSC000002,27.5,16.2,5.3,46,13.2,61,68,11.4,0.86,0.02,0.36,24,4,193,6.1,1.7,16,0.98,160,14.6,17.6,9,1084,75,65,153,citricos,secano,si,no,ecologico,0
3,CSC000003,22.2,37.9,14.4,64,31.3,77,141,0.0,0.56,0.35,0.43,8,3,121,6.5,2.2,24,0.35,541,12.2,12.9,0,957,48,61,66,horticola,secano,no,no,convencional,0
4,CSC000004,21.6,26.8,8.7,89,22.4,39,40,10.5,0.52,0.15,0.46,40,5,251,5.9,3.7,23,0.51,267,9.9,23.1,26,764,38,71,64,olivo,aspersion,si,si,integrado,0


<a name="su-06-hospital-universitario-costa"></a>
## SU-06 · Hospital Universitario Costa

El Hospital Universitario Costa es un gran hospital público de tercer nivel, con cerca de 700 camas y vinculación universitaria, que atiende a un área de referencia amplia y concentra las especialidades más complejas de la región. Su Servicio de Gestión Clínica vela por la eficiencia y la calidad asistencial, y rinde cuentas mediante indicadores entre los que el reingreso no programado a 30 días ocupa un lugar destacado: además de su coste, se interpreta como una señal de calidad del alta y de la continuidad asistencial. El hospital dispone de historia clínica electrónica y de un incipiente programa de enfermería de enlace para el seguimiento posalta, pero ese recurso es escaso y hoy se asigna sin un criterio de riesgo claro. La dirección quiere focalizarlo donde más impacto tenga y recurre a Core Analytics para conseguirlo.

> **📁 Fichero de datos**
> `SU06_salud_reingreso.csv`  ·  **Variable objetivo:** `reingreso_30d`

### La cuestión

Quiere **identificar al alta a los pacientes con alto riesgo de reingreso** en 30 días para reforzar su seguimiento y focalizar el recurso de enfermería de enlace.

### Diccionario de variables

| Identificador | Significado | Unidades |
|---|---|---|
| `edad` | Edad del paciente | años |
| `dias_ingreso` | Duración del ingreso | días |
| `num_diagnosticos` | Diagnósticos registrados | nº |
| `num_procedimientos` | Procedimientos realizados | nº |
| `num_medicamentos` | Medicamentos al alta | nº |
| `num_medicamentos_alto_riesgo` | Medicamentos de alto riesgo | nº |
| `num_ingresos_previos_12m` | Ingresos en los 12 meses previos | nº |
| `num_urgencias_previas_12m` | Visitas a urgencias previas (12 m) | nº |
| `indice_comorbilidad` | Índice de comorbilidad de Charlson | 0-15 |
| `num_comorbilidades` | Número de comorbilidades | nº |
| `hemoglobina` | Hemoglobina | g/dL |
| `creatinina` | Creatinina | mg/dL |
| `sodio_meq_l` | Sodio sérico | mEq/L |
| `albumina_g_dl` | Albúmina | g/dL |
| `glucosa_mg_dl` | Glucemia | mg/dL |
| `presion_sistolica` | Presión arterial sistólica | mmHg |
| `frecuencia_cardiaca` | Frecuencia cardiaca | lpm |
| `saturacion_o2_pct` | Saturación de oxígeno | % |
| `imc` | Índice de masa corporal | kg/m² |
| `num_visitas_ap_previas` | Visitas a atención primaria (12 m) | nº |
| `dias_desde_ultimo_ingreso` | Días desde el último ingreso | días |
| `coste_ingreso_eur` | Coste estimado del ingreso | € |
| `barthel_index` | Índice de Barthel (autonomía) | 0-100 |
| `num_especialistas_implicados` | Especialistas implicados | nº |
| `servicio` | Servicio de alta | categórico (cardiologia/cirugia/medicina_interna/neumologia/nefrologia/digestivo) |
| `tipo_alta` | Tipo de alta | categórico (domicilio/traslado/voluntaria) |
| `sexo` | Sexo biológico | categórico (M/F) |
| `tipo_ingreso` | Tipo de ingreso | categórico (programado/urgente) |
| `vive_solo` | Vive solo | categórico (si/no) |
| `soporte_social` | Soporte social | categórico (bajo/medio/alto) |
| **`reingreso_30d`** | **Reingreso no programado a 30 días (objetivo)** | binario (0=No/1=Si) |

### Preguntas que el cliente desea responder

1. ¿Con qué fiabilidad puede predecirse el reingreso a 30 días?
2. ¿Qué factores se asocian a mayor riesgo (ingresos previos, comorbilidad, edad, nº de medicamentos)?
3. ¿Difieren el riesgo y sus causas entre servicios (medicina interna, nefrología…)?
4. ¿Qué peso tiene el tipo de alta (voluntaria, traslado) en el riesgo?
5. Si solo puede seguirse al 20 % de las altas, ¿a qué pacientes priorizar y qué porcentaje de reingresos se cubriría?
6. ¿Qué umbral de riesgo define el grupo de "seguimiento intensivo"?
7. ¿El modelo se comporta de forma equitativa por sexo y por edad?

### Líneas abiertas *(políticas, protocolos y despliegue)*

- Definir un **protocolo de seguimiento posalta escalonado** por nivel de riesgo (llamada, cita precoz, conciliación de la medicación).
- Establecer una **política de asignación del recurso de enfermería de enlace** según el riesgo.
- **Herramienta de despliegue**: integrar el cálculo del riesgo en el momento del alta, con una lista de trabajo para el equipo de seguimiento.
- Fijar un **plan de monitorización y de evaluación de impacto** (reingresos evitados) del programa.

In [12]:
# SU-06 · Hospital Universitario Costa
df_su06 = cargar("SU06_salud_reingreso.csv")
print("\nReparto del objetivo:")
print(df_su06["reingreso_30d"].value_counts(normalize=True).round(3))
df_su06.head()

SU06_salud_reingreso.csv  ←  GitHub
  13,000 filas × 32 columnas
  Valores faltantes: 0
  Duplicados: 0
  ⚠️  Identificador(es) no predictivo(s): id_episodio → excluir del modelo

Reparto del objetivo:
reingreso_30d
0    0.859
1    0.141
Name: proportion, dtype: float64


,id_episodio,edad,dias_ingreso,num_diagnosticos,num_procedimientos,num_medicamentos,num_medicamentos_alto_riesgo,num_ingresos_previos_12m,num_urgencias_previas_12m,indice_comorbilidad,num_comorbilidades,hemoglobina,creatinina,sodio_meq_l,albumina_g_dl,glucosa_mg_dl,presion_sistolica,frecuencia_cardiaca,saturacion_o2_pct,imc,num_visitas_ap_previas,dias_desde_ultimo_ingreso,coste_ingreso_eur,barthel_index,num_especialistas_implicados,servicio,tipo_alta,sexo,tipo_ingreso,vive_solo,soporte_social,reingreso_30d
0,REH000000,74,6,3,2,8,2,0,0,5.2,0,14.6,0.40,136.2,3.9,60,116,82,100,23.2,0,48,1533,82,2,nefrologia,domicilio,F,urgente,no,alto,0
1,REH000001,62,12,6,1,12,0,0,1,3.6,1,11.6,0.44,136.1,3.7,156,139,68,93,30.0,3,320,1280,81,3,cardiologia,domicilio,F,urgente,no,medio,1
2,REH000002,85,7,1,1,12,3,3,0,3.8,5,11.6,0.62,141.0,3.8,108,169,74,95,16.8,6,463,6477,73,2,nefrologia,domicilio,M,programado,no,bajo,1
3,REH000003,71,2,6,3,5,2,2,2,0.4,2,13.0,0.40,142.1,3.4,96,126,78,96,29.3,13,118,10709,90,2,cirugia,voluntaria,F,urgente,no,alto,0
4,REH000004,60,4,6,2,7,2,1,0,1.7,1,12.6,0.41,143.7,3.1,65,147,92,98,30.2,3,431,3572,70,3,cirugia,domicilio,M,urgente,si,alto,0
